# StringSense Complete ABSA Pipeline Notebook

This notebook provides a complete end-to-end pipeline for the StringSense review analysis workflow.

It includes:

1. Environment setup
2. Latest archive loading
3. `jieba` tokenization and custom dictionary loading
4. Normalization rules loading
5. Rule-based aspect signal extraction
6. Patched practical matrix generation
7. TF-IDF mention model training
8. TF-IDF sentiment model training
9. Full-corpus inference
10. TF-IDF string feature matrix generation
11. Comparison with the official practical matrix

All markdown and code comments are written in English only.

## 1. Install Required Packages

Run this cell first if your environment does not already have the required libraries.

In [ ]:
# Install the required packages if needed
# You may comment out this cell after the packages are installed.

# !pip install jieba scikit-learn pandas numpy openpyxl matplotlib wordcloud joblib

## 2. Import Libraries and Define Paths

In [ ]:
import json
import math
import re
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import joblib

BASE_DIR = Path(".").resolve()

ARCHIVE_ZIP = BASE_DIR / "data" / "归档.zip"
DICT_CSV = BASE_DIR / "data" / "domain_dictionary_optimized_v6.csv"
NORM_CSV = BASE_DIR / "data" / "normalization_rules_v6.csv"

MENTION_DATA_CSV = BASE_DIR / "data" / "nlp_absa_long_dataset_latest.csv"
SENTIMENT_DATA_CSV = BASE_DIR / "data" / "nlp_absa_high_confidence_latest.csv"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

## 3. Load Latest Raw Review Data from the Archive

In [ ]:
with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
    with z.open("badminton_strings_data.json") as f:
        raw_data = json.load(f)

strings = raw_data["strings"]
print("Number of strings:", len(strings))

raw_reviews = []
for item in strings:
    string_name = item.get("name", "")
    brand = item.get("brand", "")
    price_rm = item.get("price_rm", np.nan)
    for r in item.get("reviews", []):
        text = (r.get("content") or "").strip()
        if not text:
            continue
        raw_reviews.append({
            "string_name": string_name,
            "brand": brand,
            "price_rm": price_rm,
            "review_id": r.get("review_id", ""),
            "review_text": text,
            "likes_count": r.get("likes", 0) or 0,
            "rating_label": r.get("rating_label", "")
        })

reviews_df = pd.DataFrame(raw_reviews)
print("Number of raw reviews:", len(reviews_df))
reviews_df.head()

## 4. Load the Optimized Domain Dictionary and Normalization Rules

In [ ]:
dict_df = pd.read_csv(DICT_CSV)
norm_df = pd.read_csv(NORM_CSV)

print("Dictionary rows:", len(dict_df))
print("Normalization rows:", len(norm_df))

dict_df.head()

## 5. Register the Custom Dictionary into jieba

This step allows the tokenizer to keep badminton-specific terms intact.

In [ ]:
custom_terms = (
    dict_df["term"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

for term in custom_terms:
    if term:
        jieba.add_word(term)

print("Custom terms registered into jieba:", len(custom_terms))

## 6. Define Text Normalization and Clause Splitting Functions

In [ ]:
normalization_rules = list(zip(norm_df["pattern"].astype(str).tolist(), norm_df["replacement"].astype(str).tolist()))

def normalize_text(text: str) -> str:
    text = str(text)
    for pattern, replacement in normalization_rules:
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_clauses(text: str):
    text = normalize_text(text)
    parts = re.split(r"[。！？；;.!?\n]+|但是|但|不过|然而|就是|而且|同时", text)
    parts = [p.strip(" ，,：:、 ") for p in parts if p and p.strip(" ，,：:、 ")]
    return parts

reviews_df["normalized_text"] = reviews_df["review_text"].apply(normalize_text)
reviews_df["clauses"] = reviews_df["normalized_text"].apply(split_into_clauses)

reviews_df[["string_name", "review_text", "normalized_text", "clauses"]].head()

## 7. Build Aspect Lexicons from the Dictionary File

In [ ]:
aspect_lexicon = defaultdict(lambda: {"aspect_terms": set(), "positive_terms": set(), "negative_terms": set()})

for _, row in dict_df.iterrows():
    aspect = str(row["aspect"]).strip()
    term_type = str(row["term_type"]).strip()
    term = str(row["term"]).strip()
    polarity = str(row["polarity"]).strip().lower()

    if not aspect or not term or term == "nan":
        continue

    if term_type == "aspect_term":
        aspect_lexicon[aspect]["aspect_terms"].add(term)

    if polarity == "positive":
        aspect_lexicon[aspect]["positive_terms"].add(term)
    elif polarity == "negative":
        aspect_lexicon[aspect]["negative_terms"].add(term)

list(aspect_lexicon.keys())

## 8. Rule-Based Aspect Signal Extraction

This stage produces review-clause-level aspect signals.

In [ ]:
def detect_clause_aspect_polarity(clause: str, aspect: str):
    lex = aspect_lexicon[aspect]

    aspect_hits = [t for t in lex["aspect_terms"] if t in clause]
    pos_hits = [t for t in lex["positive_terms"] if t in clause]
    neg_hits = [t for t in lex["negative_terms"] if t in clause]

    has_signal = bool(aspect_hits or pos_hits or neg_hits)
    if not has_signal:
        return None

    if len(pos_hits) > len(neg_hits):
        polarity = "positive"
    elif len(neg_hits) > len(pos_hits):
        polarity = "negative"
    else:
        polarity = "neutral"

    return {
        "aspect": aspect,
        "polarity": polarity,
        "aspect_hits": aspect_hits,
        "positive_hits": pos_hits,
        "negative_hits": neg_hits
    }

signal_rows = []
for _, row in reviews_df.iterrows():
    for clause in row["clauses"]:
        for aspect in aspect_lexicon.keys():
            result = detect_clause_aspect_polarity(clause, aspect)
            if result is None:
                continue

            signal_rows.append({
                "string_name": row["string_name"],
                "brand": row["brand"],
                "price_rm": row["price_rm"],
                "review_id": row["review_id"],
                "likes_count": row["likes_count"],
                "rating_label": row["rating_label"],
                "clause": clause,
                "aspect": result["aspect"],
                "polarity": result["polarity"],
                "aspect_hits": "|".join(result["aspect_hits"]),
                "positive_hits": "|".join(result["positive_hits"]),
                "negative_hits": "|".join(result["negative_hits"]),
            })

signals_df = pd.DataFrame(signal_rows)
print("Review-aspect signal rows:", len(signals_df))
signals_df.head()

## 9. Save the Rule-Based Review-Aspect Signals

In [ ]:
signals_csv = OUTPUT_DIR / "rule_based_review_aspect_signals.csv"
signals_df.to_csv(signals_csv, index=False, encoding="utf-8-sig")
signals_csv

## 10. Generate a Patched Practical Matrix

This matrix is review-driven and recommendation-oriented.
The score is not a physical measurement. It is a smoothed, evidence-based aspect signal.

In [ ]:
aspect_list = sorted(aspect_lexicon.keys())

priors = {}
for aspect in aspect_list:
    subset = signals_df[signals_df["aspect"] == aspect]
    pos = (subset["polarity"] == "positive").sum()
    neg = (subset["polarity"] == "negative").sum()
    priors[aspect] = (pos + 1.0) / (pos + neg + 2.0) if (pos + neg) > 0 else 0.5

def evidence_weight(likes_count):
    return 1.0 + 0.12 * math.log1p(float(likes_count))

matrix_rows = []
for string_name, sdf in signals_df.groupby("string_name"):
    row_out = {"string_name": string_name}
    price_values = reviews_df.loc[reviews_df["string_name"] == string_name, "price_rm"].dropna()
    row_out["price_rm"] = float(price_values.iloc[0]) if len(price_values) > 0 else np.nan

    for aspect in aspect_list:
        a_df = sdf[sdf["aspect"] == aspect].copy()
        if len(a_df) == 0:
            raw_score = priors[aspect]
            confidence = 0.0
        else:
            a_df["weight"] = a_df["likes_count"].apply(evidence_weight)
            pos_weight = a_df.loc[a_df["polarity"] == "positive", "weight"].sum()
            neg_weight = a_df.loc[a_df["polarity"] == "negative", "weight"].sum()
            evidence = pos_weight + neg_weight

            alpha = 8.0
            if aspect in {"attack", "control", "elasticity", "sound"}:
                alpha = 6.0

            raw_score = (pos_weight + alpha * priors[aspect]) / (evidence + alpha) if evidence > 0 else priors[aspect]
            confidence = min(1.0, evidence / 25.0)

        row_out[f"{aspect}_review_raw"] = float(raw_score)
        row_out[f"{aspect}_confidence"] = float(confidence)

    matrix_rows.append(row_out)

practical_matrix_df = pd.DataFrame(matrix_rows)

if "string_movement_review_raw" in practical_matrix_df.columns:
    practical_matrix_df["string_movement_review_raw"] = 1.0 - practical_matrix_df["string_movement_review_raw"]

if "value_for_money_review_raw" in practical_matrix_df.columns:
    if practical_matrix_df["price_rm"].notna().sum() > 0:
        min_price = practical_matrix_df["price_rm"].min()
        max_price = practical_matrix_df["price_rm"].max()
        if pd.notna(min_price) and pd.notna(max_price) and max_price > min_price:
            price_affordability = 1.0 - ((practical_matrix_df["price_rm"] - min_price) / (max_price - min_price))
            practical_matrix_df["value_for_money"] = 0.75 * practical_matrix_df["value_for_money_review_raw"] + 0.25 * price_affordability.fillna(0.5)
        else:
            practical_matrix_df["value_for_money"] = practical_matrix_df["value_for_money_review_raw"]
    else:
        practical_matrix_df["value_for_money"] = practical_matrix_df["value_for_money_review_raw"]

for aspect in aspect_list:
    src = f"{aspect}_review_raw"
    if aspect == "value_for_money":
        continue
    practical_matrix_df[aspect] = practical_matrix_df[src]

practical_matrix_df["beginner_fit_score"] = (
    0.35 * practical_matrix_df.get("comfort", 0.5) +
    0.25 * practical_matrix_df.get("control", 0.5) +
    0.20 * practical_matrix_df.get("durability", 0.5) +
    0.20 * practical_matrix_df.get("value_for_money", 0.5)
)

practical_matrix_df["stability_score"] = practical_matrix_df.get("string_movement", 0.5)
practical_matrix_df["all_round_score"] = practical_matrix_df[[c for c in ["attack", "comfort", "control", "durability", "elasticity", "sound", "stability_score", "tension_retention", "value_for_money"] if c in practical_matrix_df.columns]].mean(axis=1)

practical_matrix_df = practical_matrix_df.sort_values("string_name").reset_index(drop=True)
practical_matrix_df.head()

## 11. Save the Patched Practical Matrix

In [ ]:
practical_matrix_csv = OUTPUT_DIR / "patched_practical_string_feature_matrix.csv"
practical_matrix_xlsx = OUTPUT_DIR / "patched_practical_string_feature_matrix.xlsx"

practical_matrix_df.to_csv(practical_matrix_csv, index=False, encoding="utf-8-sig")
practical_matrix_df.to_excel(practical_matrix_xlsx, index=False)

practical_matrix_csv, practical_matrix_xlsx

## 12. Prepare TF-IDF Training Data

The mention model uses `nlp_absa_long_dataset_latest.csv`.
The sentiment model uses `nlp_absa_high_confidence_latest.csv`.

In [ ]:
mention_df = pd.read_csv(MENTION_DATA_CSV)
sentiment_df = pd.read_csv(SENTIMENT_DATA_CSV)

print("Mention rows:", len(mention_df))
print("Sentiment rows:", len(sentiment_df))

mention_df.head(2)

## 13. Define jieba-Based Tokenization for TF-IDF

In [ ]:
stopwords = {
    "这个", "那个", "真的", "感觉", "觉得", "还是", "就是", "因为", "然后", "而且",
    "如果", "所以", "但是", "不过", "一个", "一种", "一下", "一些", "没有", "不是",
    "比较", "非常", "特别", "很多", "有点", "一点", "已经", "时候", "东西", "问题",
    "方面", "可能", "这样", "那样", "我们", "你们", "他们"
}
single_char_whitelist = {"脆", "响", "闷", "弹", "硬", "软", "稳", "顶", "炸"}

def tokenize_for_tfidf(text: str):
    text = normalize_text(text)
    tokens = []
    for tok in jieba.lcut(text):
        tok = tok.strip()
        if not tok:
            continue
        if tok in stopwords:
            continue
        if tok.isdigit():
            continue
        if len(tok) == 1 and tok not in single_char_whitelist:
            continue
        tokens.append(tok)
    return tokens

def build_model_input(review_text: str, aspect: str):
    tokens = tokenize_for_tfidf(review_text)
    return f"aspect: {aspect} [SEP] " + " \".join(tokens)

## 14. Train the TF-IDF Mention Model

In [ ]:
mention_df = mention_df.copy()
mention_df["text_input"] = mention_df.apply(lambda x: build_model_input(x["review_text"], x["aspect"]), axis=1)

train_mention = mention_df[mention_df["split"] == "train"].copy()
test_mention = mention_df[mention_df["split"] == "test"].copy()

X_train_mention = train_mention["text_input"].tolist()
y_train_mention = train_mention["mention_flag"].astype(int).tolist()

X_test_mention = test_mention["text_input"].tolist()
y_test_mention = test_mention["mention_flag"].astype(int).tolist()

mention_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, max_features=30000)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear"))
])

mention_pipeline.fit(X_train_mention, y_train_mention)
mention_pred = mention_pipeline.predict(X_test_mention)

mention_metrics = {
    "accuracy": accuracy_score(y_test_mention, mention_pred),
    "macro_f1": f1_score(y_test_mention, mention_pred, average="macro")
}
mention_metrics

## 15. Train the TF-IDF Sentiment Model

In [ ]:
sentiment_df = sentiment_df.copy()
sentiment_df = sentiment_df[sentiment_df["mention_flag"] == 1].copy()
sentiment_df = sentiment_df[sentiment_df["sentiment_id"].isin([-1.0, 1.0])].copy()
sentiment_df["y_sentiment"] = sentiment_df["sentiment_id"].map({-1.0: 0, 1.0: 1})
sentiment_df["text_input"] = sentiment_df.apply(lambda x: build_model_input(x["review_text"], x["aspect"]), axis=1)

train_sent = sentiment_df[sentiment_df["split"] == "train"].copy()
test_sent = sentiment_df[sentiment_df["split"] == "test"].copy()

X_train_sent = train_sent["text_input"].tolist()
y_train_sent = train_sent["y_sentiment"].astype(int).tolist()

X_test_sent = test_sent["text_input"].tolist()
y_test_sent = test_sent["y_sentiment"].astype(int).tolist()

sentiment_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2, max_features=30000)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear"))
])

sentiment_pipeline.fit(X_train_sent, y_train_sent)
sentiment_pred = sentiment_pipeline.predict(X_test_sent)

sentiment_metrics = {
    "accuracy": accuracy_score(y_test_sent, sentiment_pred),
    "macro_f1": f1_score(y_test_sent, sentiment_pred, average="macro")
}
sentiment_metrics

## 16. Save the TF-IDF Models

In [ ]:
mention_model_path = OUTPUT_DIR / "tfidf_mention_model.joblib"
sentiment_model_path = OUTPUT_DIR / "tfidf_sentiment_model.joblib"

joblib.dump(mention_pipeline, mention_model_path)
joblib.dump(sentiment_pipeline, sentiment_model_path)

mention_model_path, sentiment_model_path

## 17. Run Full-Corpus TF-IDF Inference

This stage predicts mention and sentiment for all raw reviews and all aspects.

In [ ]:
full_rows = []
for _, row in reviews_df.iterrows():
    for aspect in aspect_list:
        text_input = build_model_input(row["normalized_text"], aspect)
        mention_hat = int(mention_pipeline.predict([text_input])[0])

        sentiment_hat = None
        if mention_hat == 1:
            sentiment_hat = int(sentiment_pipeline.predict([text_input])[0])

        full_rows.append({
            "string_name": row["string_name"],
            "brand": row["brand"],
            "price_rm": row["price_rm"],
            "review_id": row["review_id"],
            "review_text": row["normalized_text"],
            "likes_count": row["likes_count"],
            "aspect": aspect,
            "mention_pred": mention_hat,
            "sentiment_pred": sentiment_hat
        })

tfidf_full_df = pd.DataFrame(full_rows)
print("Full TF-IDF review-aspect rows:", len(tfidf_full_df))
tfidf_full_df.head()

## 18. Aggregate the TF-IDF String Feature Matrix

In [ ]:
tfidf_matrix_rows = []
for string_name, sdf in tfidf_full_df.groupby("string_name"):
    row_out = {"string_name": string_name}
    price_values = reviews_df.loc[reviews_df["string_name"] == string_name, "price_rm"].dropna()
    row_out["price_rm"] = float(price_values.iloc[0]) if len(price_values) > 0 else np.nan

    for aspect in aspect_list:
        a_df = sdf[(sdf["aspect"] == aspect) & (sdf["mention_pred"] == 1)].copy()
        if len(a_df) == 0:
            score = 0.5
        else:
            pos = (a_df["sentiment_pred"] == 1).sum()
            neg = (a_df["sentiment_pred"] == 0).sum()
            score = (pos + 1.0) / (pos + neg + 2.0) if (pos + neg) > 0 else 0.5
        row_out[aspect] = float(score)

    row_out["string_movement"] = 1.0 - row_out["string_movement"]

    if pd.notna(row_out["price_rm"]):
        all_prices = practical_matrix_df["price_rm"].dropna()
        if len(all_prices) > 0 and all_prices.max() > all_prices.min():
            affordability = 1.0 - ((row_out["price_rm"] - all_prices.min()) / (all_prices.max() - all_prices.min()))
            row_out["value_for_money"] = 0.75 * row_out["value_for_money"] + 0.25 * affordability

    tfidf_matrix_rows.append(row_out)

tfidf_matrix_df = pd.DataFrame(tfidf_matrix_rows).sort_values("string_name").reset_index(drop=True)
tfidf_matrix_df.head()

## 19. Save TF-IDF Outputs

In [ ]:
tfidf_review_csv = OUTPUT_DIR / "tfidf_full_review_aspect_predictions.csv"
tfidf_matrix_csv = OUTPUT_DIR / "tfidf_string_feature_matrix.csv"
tfidf_matrix_xlsx = OUTPUT_DIR / "tfidf_string_feature_matrix.xlsx"

tfidf_full_df.to_csv(tfidf_review_csv, index=False, encoding="utf-8-sig")
tfidf_matrix_df.to_csv(tfidf_matrix_csv, index=False, encoding="utf-8-sig")
tfidf_matrix_df.to_excel(tfidf_matrix_xlsx, index=False)

tfidf_review_csv, tfidf_matrix_csv

## 20. Compare the Official Practical Matrix and the TF-IDF Matrix

In [ ]:
compare_df = practical_matrix_df[["string_name"] + [c for c in aspect_list if c in practical_matrix_df.columns]].merge(
    tfidf_matrix_df[["string_name"] + aspect_list],
    on="string_name",
    suffixes=("_practical", "_tfidf")
)

for aspect in aspect_list:
    compare_df[f"{aspect}_abs_diff"] = (compare_df[f"{aspect}_practical"] - compare_df[f"{aspect}_tfidf"]).abs()

compare_df = compare_df.sort_values([f"{aspect_list[0]}_abs_diff"], ascending=False).reset_index(drop=True)

compare_csv = OUTPUT_DIR / "practical_vs_tfidf_comparison.csv"
compare_df.to_csv(compare_csv, index=False, encoding="utf-8-sig")

compare_df.head()

## 21. Save a Run Summary

In [ ]:
summary = {
    "strings_count": int(len(strings)),
    "raw_reviews_count": int(len(reviews_df)),
    "rule_based_signal_rows": int(len(signals_df)),
    "practical_matrix_rows": int(len(practical_matrix_df)),
    "tfidf_full_rows": int(len(tfidf_full_df)),
    "tfidf_matrix_rows": int(len(tfidf_matrix_df)),
    "mention_accuracy": float(mention_metrics["accuracy"]),
    "mention_macro_f1": float(mention_metrics["macro_f1"]),
    "sentiment_accuracy": float(sentiment_metrics["accuracy"]),
    "sentiment_macro_f1": float(sentiment_metrics["macro_f1"]),
}

summary_path = OUTPUT_DIR / "run_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

## 22. Final Notes

Use the following outputs as your main references:

- `patched_practical_string_feature_matrix.csv` for the recommendation system
- `tfidf_string_feature_matrix.csv` for the model baseline
- `practical_vs_tfidf_comparison.csv` for analysis and comparison

This notebook intentionally keeps both:
- a review-driven practical matrix
- a model-based TF-IDF baseline

so that your system and your experiments can be separated clearly.